# [[FILL: investigation title]]

One dated investigation notebook in a repeatable series -- see `notebooks/experiments/README.md` for the index and workflow. Copied from `_template.ipynb`; never edited in place once committed.

**Purpose.** *[[FILL: one or two sentences on what this investigation establishes]]*

**Non-goals.** This notebook does not redesign the kernel, implement newly-discovered behavior, or modify telemetry. Sweep orchestration stays notebook-local unless a repeated need for shared code shows up across investigations.

In [ ]:
from __future__ import annotations

import copy
import os
import subprocess
import tempfile
from dataclasses import asdict
from datetime import datetime, timezone

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import yaml
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.model_selection import KFold, cross_val_score

from simlab._merge import deep_merge
from simlab.runner import SCHEMA_VERSION, RunRequest, execute_run

%matplotlib inline
pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 160)
plt.rcParams["figure.dpi"] = 100

## Reproducibility header

Checkpoint for which kernel version produced the results below. **Rule:** only commit this notebook once `kernel_commit_on_main` prints `True` (see `notebooks/experiments/README.md` for why).

In [ ]:
def _git(*args: str) -> str:
    return subprocess.run(
        ["git", *args], capture_output=True, text=True, check=True
    ).stdout.strip()


def _is_ancestor_of_main(commit: str) -> bool:
    result = subprocess.run(
        ["git", "merge-base", "--is-ancestor", commit, "main"],
        capture_output=True,
        text=True,
    )
    return result.returncode == 0


def _is_clean(pathspec: str) -> bool:
    return _git("status", "--porcelain", "--", pathspec) == ""


git_branch = _git("rev-parse", "--abbrev-ref", "HEAD")
git_commit = _git("rev-parse", "HEAD")
git_dirty = bool(_git("status", "--porcelain"))

# The commit that actually determines kernel behavior, as opposed to
# git_commit (HEAD), which also includes this notebook's own not-yet-merged
# commit and any other unrelated in-progress work on this branch. The ":/"
# pathspec prefix anchors to the repo root regardless of the kernel's cwd
# (notebooks/experiments/), which a plain "src/simlab" would silently miss.
kernel_commit = _git("log", "-1", "--format=%H", "--", ":/src/simlab")
# git log only inspects committed history -- an uncommitted edit under
# src/simlab wouldn't change kernel_commit at all, so it has to be checked
# for separately. git_dirty (whole-repo) isn't specific enough: this
# notebook itself is normally dirty while being edited, which shouldn't
# block the check, but a dirty src/simlab always should.
kernel_path_clean = _is_clean(":/src/simlab")
kernel_commit_on_main = (
    bool(kernel_commit) and kernel_path_clean and _is_ancestor_of_main(kernel_commit)
)

# EDIT: match this file's own name (notebooks/experiments/<date>-<slug>.ipynb).
EXPERIMENT_NAME = "EDIT-ME-date-slug"

print("Notebook reproducibility header")
print(f"  experiment_name      : {EXPERIMENT_NAME}")
print(f"  executed_at_utc      : {datetime.now(timezone.utc).isoformat()}")
print(f"  git_branch           : {git_branch}")
print(f"  git_commit           : {git_commit}")
print(f"  git_dirty            : {git_dirty}")
print(f"  kernel_commit        : {kernel_commit}")
print(f"  kernel_path_clean    : {kernel_path_clean}")
print(f"  kernel_commit_on_main: {kernel_commit_on_main}")
print(f"  schema_version       : {SCHEMA_VERSION}")

if not kernel_path_clean:
    print()
    print("WARNING: src/simlab has uncommitted changes. The code actually")
    print("exercised by this run may not match kernel_commit at all -- commit")
    print("or discard those changes and rerun this cell before trusting")
    print("kernel_commit_on_main.")
elif not kernel_commit_on_main:
    print()
    print("WARNING: kernel_commit is not reachable from main. Do not commit this")
    print("notebook to notebooks/experiments/ until the kernel code it exercised")
    print("has actually merged to main -- rerun this cell after that merge to")
    print("confirm a stable, permanent commit reference.")

## Sweep design

*[[FILL: the baseline scenario, and why each axis below was chosen]]*

Axes are one-factor-at-a-time (OFAT) from a single baseline to stay interpretable; a full grid is reserved for a parameter pair with a clear a-priori interaction hypothesis. Each scenario runs `len(SEEDS)` replicates so scenario effects can be separated from stochastic ones. Optional additions not scaffolded here -- interaction grids, structural/boundary-condition scenarios, mechanism-activation or trajectory-diversity diagnostics -- follow the same patterns; see a prior notebook in this series for worked examples.

In [ ]:
STEPS = 300  # EDIT: enough ticks for the slowest scenario to settle or stabilize
SEEDS = list(range(10))  # EDIT: replicates per scenario

# EDIT: starting scenario. Kept close to configs/default.yaml as a default.
BASELINE_CFG = {
    "world": {
        "rng_seed": 0,
        "truths": {0: True},
        "noise": {"OBSERVE": 0.1, "HEAR": 0.15, "VERIFY": 0.05},
        "observation": {"private_event_rate": 0.1, "global_event_rate": 0.0},
    },
    "agent": {
        "defaults": {
            "observation": {"attention": 1.0, "bias": 0.0},
            "trust": {"default": 0.5},
            "social": {
                "confidence_bound": 1.0,
                "trust_update_rate": 0.0,
                "update_trust_on_rejection": True,
            },
            "learning": {
                "rate": 0.1,
                "observe_weight": 0.6,
                "hear_weight": 0.3,
                "verify_weight": 1.0,
            },
            "action_preference": {
                "IDLE": 0.0,
                "VERIFY": 0.9,
                "COMMUNICATE": 0.7,
                "BROADCAST": 0.5,
            },
            "action_cost": {
                "IDLE": 0.0,
                "VERIFY": 0.35,
                "COMMUNICATE": 0.15,
                "BROADCAST": 0.30,
            },
        },
        "profiles": [{"name": "default", "count": 60}],
    },
}

# EDIT: replace these two example axes with the ones this investigation needs.
# Each entry is (label, override_dict), deep-merged onto BASELINE_CFG.
OFAT_AXES: dict[str, list[tuple[str, dict]]] = {
    "example_axis_confidence_bound": [  # baseline: 1.0 (unbounded acceptance)
        (
            "narrow_0.05",
            {"agent": {"defaults": {"social": {"confidence_bound": 0.05}}}},
        ),
        (
            "moderate_0.3",
            {"agent": {"defaults": {"social": {"confidence_bound": 0.3}}}},
        ),
    ],
    "example_axis_noise_regime": [  # baseline: OBSERVE=0.1 HEAR=0.15 VERIFY=0.05
        ("none", {"world": {"noise": {"OBSERVE": 0.0, "HEAR": 0.0, "VERIFY": 0.0}}}),
        ("high", {"world": {"noise": {"OBSERVE": 0.3, "HEAR": 0.35, "VERIFY": 0.2}}}),
    ],
}


def build_scenarios() -> list[dict]:
    scenarios = [
        {
            "id": "baseline",
            "group": "baseline",
            "swept_axis": None,
            "label": "baseline",
            "cfg": BASELINE_CFG,
        }
    ]

    for axis, variants in OFAT_AXES.items():
        for label, overrides in variants:
            cfg = deep_merge(copy.deepcopy(BASELINE_CFG), overrides)
            scenarios.append(
                {
                    "id": f"{axis}:{label}",
                    "group": "ofat",
                    "swept_axis": axis,
                    "label": label,
                    "cfg": cfg,
                }
            )

    # EDIT: append interaction-grid / structural scenarios here if needed,
    # following the same {"id", "group", "swept_axis", "label", "cfg"} shape.

    return scenarios


SCENARIOS = build_scenarios()
print(
    f"{len(SCENARIOS)} unique scenario configs x {len(SEEDS)} seeds = {len(SCENARIOS) * len(SEEDS)} runs"
)

## Executing the sweep

Each scenario is written to a temp YAML and run through `execute_run()` unmodified. Artifacts aren't persisted to `runs/`, keeping the repo clean -- `result.scenario`/`result.summary` already carry what's needed. Full telemetry is kept only for `seed=0` per scenario, for any trajectory plots.

In [ ]:
records = []
trajectories: dict[str, list] = {}  # scenario_id -> telemetry for seed 0

with tempfile.TemporaryDirectory() as tmpdir:
    for scenario in SCENARIOS:
        for seed in SEEDS:
            cfg = deep_merge(
                copy.deepcopy(scenario["cfg"]), {"world": {"rng_seed": seed}}
            )
            safe_name = (
                scenario["id"]
                .replace("/", "_")
                .replace(":", "_")
                .replace(",", "_")
                .replace("=", "")
            )
            path = os.path.join(tmpdir, f"{safe_name}_{seed}.yaml")
            with open(path, "w") as f:
                yaml.safe_dump(cfg, f)

            result = execute_run(RunRequest(config_path=path, steps=STEPS))

            records.append(
                {
                    "scenario_id": scenario["id"],
                    "scenario_group": scenario["group"],
                    "swept_axis": scenario["swept_axis"],
                    "label": scenario["label"],
                    "seed": seed,
                    "run_id": result.metadata.run_id,
                    "scenario_fingerprint": result.metadata.scenario_fingerprint,
                    "run_spec_fingerprint": result.metadata.run_spec_fingerprint,
                    **result.scenario,
                    **asdict(result.summary),
                }
            )

            if seed == 0:
                trajectories[scenario["id"]] = result.telemetry

runs = pd.DataFrame(records)
print(f"Analysis dataset: {runs.shape[0]} runs x {runs.shape[1]} columns")
runs.head()

## Outcome diversity

Label rates across all runs. *[[FILL: split `runs` first if there's a meaningful sub-grouping worth comparing; otherwise remove this sentence]]*

In [ ]:
label_cols = [
    "converged",
    "final_consensus",
    "final_truth_aligned",
    "final_false_consensus",
]
print(f"Label rates across all {len(runs)} runs:")
print(runs[label_cols].mean().round(3).to_string())

fig, ax = plt.subplots(figsize=(6, 4))
ax.bar(range(len(label_cols)), runs[label_cols].mean())
ax.set_xticks(range(len(label_cols)))
ax.set_xticklabels(label_cols, rotation=20, ha="right")
ax.set_ylabel("rate across runs")
ax.set_title("Outcome label rates")
plt.tight_layout()
plt.show()

# EDIT: if there's a meaningful sub-grouping, split runs here, e.g.:
# truth_anchored = runs[runs.swept_axis != "some_boundary_condition"]

**Reading this:** *[[FILL: is one outcome regime dominant, or is there real diversity?]]*

## Scenario signal vs. stochastic noise

For each outcome metric, `SS_between / SS_total` (a one-way ANOVA style decomposition on `scenario_id`) separates between-scenario variance from within-scenario (seed) variance.

In [ ]:
def between_scenario_variance_share(df: pd.DataFrame, metric: str) -> float:
    sub = df[["scenario_id", metric]].dropna()
    grand_mean = sub[metric].mean()
    group_means = sub.groupby("scenario_id")[metric].mean()
    group_sizes = sub.groupby("scenario_id")[metric].size()
    ss_between = float((group_sizes * (group_means - grand_mean) ** 2).sum())
    ss_total = float(((sub[metric] - grand_mean) ** 2).sum())
    return ss_between / ss_total if ss_total else float("nan")


# EDIT: outcome metrics that matter for this investigation.
variance_metrics = [
    "final_mean_truth_error",
    "final_mean_claim_belief_variance",
    "mean_belief_volatility",
    "convergence_tick",
]
variance_shares = {
    m: between_scenario_variance_share(runs, m) for m in variance_metrics
}
for m, v in variance_shares.items():
    print(f"  {m}: {v:.3f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.bar(list(variance_shares.keys()), list(variance_shares.values()), color="tab:blue")
ax.axhline(0.5, color="gray", linestyle="--", linewidth=1)
ax.set_ylabel("SS_between / SS_total")
ax.set_title("Scenario vs seed: share of variance explained by scenario")
plt.xticks(rotation=20, ha="right")
plt.tight_layout()
plt.show()

**Reading this:** *[[FILL: is a fixed scenario's outcome reproducible across seeds, or mostly noise?]]*

## Parameter response

Mean outcome per OFAT axis (baseline + variants), error bars = std across seeds.

In [ ]:
ofat = runs[runs.scenario_group.isin(["baseline", "ofat"])]
axes_list = list(ofat.swept_axis.dropna().unique())

n_axes = len(axes_list)
ncols = min(4, n_axes) or 1
nrows = -(-n_axes // ncols)  # ceil division
fig, axs = plt.subplots(nrows, ncols, figsize=(4.5 * ncols, 4 * nrows), squeeze=False)
for ax, axis in zip(axs.flat, axes_list):
    sub = pd.concat(
        [ofat[ofat.scenario_id == "baseline"], ofat[ofat.swept_axis == axis]]
    )
    grouped = sub.groupby("label")["final_mean_truth_error"].agg(["mean", "std"])
    grouped = grouped.reindex(
        sorted(grouped.index, key=lambda label: (label != "baseline", label))
    )
    ax.bar(
        grouped.index,
        grouped["mean"],
        yerr=grouped["std"],
        capsize=3,
        color="tab:orange",
    )
    ax.set_title(axis, fontsize=10)
    ax.tick_params(axis="x", labelrotation=30, labelsize=8)
    ax.set_ylabel("final truth error")
for ax in axs.flat[n_axes:]:
    ax.axis("off")
fig.suptitle("Parameter response: final mean truth error by OFAT axis")
plt.tight_layout()
plt.show()

**Reading this:** *[[FILL: which axes move the outcome a lot, which barely move it, and does that match expectations?]]*

## Predictability with simple vs. nonlinear models

Linear/logistic regression vs. a random forest, used as landscape probes (not tuned for performance): does an additive model capture most of the predictable structure, or does a model that can represent interactions/thresholds do meaningfully better?

In [ ]:
# EDIT: feature columns relevant to the swept axes (see extract_scenario_features
# in simlab/run_analysis.py for the full set of available scenario-feature columns).
feature_cols = [
    "agent_confidence_bound_mean",
    "noise.OBSERVE",
    "noise.HEAR",
    "noise.VERIFY",
    "num_agents",
]
model_df = runs.dropna(subset=feature_cols + ["final_mean_truth_error"])
X = model_df[feature_cols].values
y = model_df["final_mean_truth_error"].values
kf = KFold(n_splits=5, shuffle=True, random_state=0)
lin_r2 = cross_val_score(LinearRegression(), X, y, cv=kf, scoring="r2")
rf_r2 = cross_val_score(
    RandomForestRegressor(n_estimators=200, random_state=0), X, y, cv=kf, scoring="r2"
)
print("final_mean_truth_error ~ swept params -- 5-fold CV R^2")
print(f"  LinearRegression: {lin_r2.mean():.3f} +- {lin_r2.std():.3f}")
print(f"  RandomForest:     {rf_r2.mean():.3f} +- {rf_r2.std():.3f}")

y_bin = model_df["converged"].astype(int).values
logit_acc = rf_acc = None
if len(np.unique(y_bin)) > 1:
    logit_acc = cross_val_score(
        LogisticRegression(max_iter=1000), X, y_bin, cv=kf, scoring="accuracy"
    )
    rf_acc = cross_val_score(
        RandomForestClassifier(n_estimators=200, random_state=0),
        X,
        y_bin,
        cv=kf,
        scoring="accuracy",
    )
    print(
        f"converged ~ swept params -- 5-fold CV accuracy (majority baseline = {max(y_bin.mean(), 1 - y_bin.mean()):.3f})"
    )
    print(f"  LogisticRegression: {logit_acc.mean():.3f}")
    print(f"  RandomForest:       {rf_acc.mean():.3f}")

fig, ax = plt.subplots(figsize=(6, 4))
labels_ = ["Linear/Logistic", "RandomForest"]
bars_x = np.arange(2)
ax.bar(
    bars_x - 0.2,
    [lin_r2.mean(), rf_r2.mean()],
    width=0.4,
    label="final_mean_truth_error (R^2)",
)
if logit_acc is not None:
    ax.bar(
        bars_x + 0.2,
        [logit_acc.mean(), rf_acc.mean()],
        width=0.4,
        label="converged (accuracy)",
    )
ax.set_xticks(bars_x)
ax.set_xticklabels(labels_)
ax.set_ylim(0, 1)
ax.legend()
ax.set_title("Simple vs nonlinear models as landscape probes")
plt.tight_layout()
plt.show()

**Reading this:** *[[FILL: does linear capture most of the structure, or does nonlinear do meaningfully better -- and any caveats on why?]]*

## Conclusions

**Behavioral outcomes.** *[[FILL: what regime(s) did runs fall into? was one dominant?]]*

**Variability.** *[[FILL: how much variance was scenario- vs. seed-driven?]]*

**Mechanisms.** *[[FILL: which parameters/mechanisms mattered, and which looked inert despite being active?]]*

**Complexity.** *[[FILL: linear vs. nonlinear predictability, interactions/thresholds, trajectory diversity]]*

**Next direction.** *[[FILL: the single most evidence-backed next step -- kernel change, telemetry gap, or run-analysis fix?]]*

## Follow-up work

Evidence-driven candidates for separate PRs, not implemented here.

**Telemetry PRs**
*[[FILL: telemetry bullets, or omit this subsection if none]]*

**Run-analysis PRs**
*[[FILL: run-analysis bullets, or omit this subsection if none]]*

**Kernel PRs**
*[[FILL: kernel bullets, or omit this subsection if none]]*

**Experiment-infrastructure PRs**
*[[FILL: experiment-infrastructure bullets, or omit this subsection if none]]*

**Whenever this notebook is committed:** add a row to `notebooks/experiments/README.md` (date, file, kernel commit, one-line finding).